# Phase 2.5 — Prior Return Mechanism + C20
GitHub → tests → FinLab → Phase 2.5 → immutable Google Drive archive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, subprocess
from pathlib import Path
from google.colab import userdata

REPO_URL = 'https://github.com/hh4832/-institutional-spot-flow-study.git'
BRANCH = 'phase25-prior-return-c20'
PROJECT_DIR = Path('/content/institutional-spot-flow-study')
DRIVE_FOLDER_ID = '1zjTMbjv-SiDhkeiIpUaidZqTYGDalold'
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/Quant_Research/institutional-spot-flow-study/phase25_outputs')
token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('請在 Colab Secrets 設定 GITHUB_TOKEN。')
git_env = os.environ.copy()
git_env.update({
    'GIT_CONFIG_COUNT': '1',
    'GIT_CONFIG_KEY_0': 'http.extraHeader',
    'GIT_CONFIG_VALUE_0': f'Authorization: Bearer {token}',
})
os.chdir('/content')
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(PROJECT_DIR)], env=git_env, check=True)
os.chdir(PROJECT_DIR)
del token
git_env.pop('GIT_CONFIG_VALUE_0', None)

In [ ]:
commit_hash = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
branch = subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip()
status = subprocess.check_output(['git', 'status', '--short'], text=True).strip()
print('Git commit:', commit_hash)
print('Branch:', branch)
print('Git status:', status or 'clean')

In [ ]:
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
tests = subprocess.run(['python', '-m', 'pytest', '-q'])
if tests.returncode != 0:
    raise RuntimeError('Tests failed；停止正式研究 run。')

In [ ]:
import finlab
try:
    finlab_token = userdata.get('FINLAB_API_TOKEN')
except Exception:
    finlab_token = None
if finlab_token:
    finlab.login(finlab_token)
else:
    finlab.login()
del finlab_token

In [ ]:
from config import StudyConfig
from data_loader import load_finlab_data
from phase25_pipeline import run_phase25_study

raw = load_finlab_data(ticker='0050')
local_root = PROJECT_DIR / 'outputs_phase25'
config = StudyConfig(
    study_mode='phase25_prior_return_mechanism',
    accumulation_windows=(1, 5, 10),
    phase25_prior_return_windows=(5, 20),
    phase25_return_horizons=(1, 2, 3, 5, 10, 20),
    output_root=local_root,
)
output_dir = run_phase25_study(raw, config, tests_passed=True)
print('Local output:', output_dir)

In [ ]:
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
destination = DRIVE_OUTPUT_ROOT / output_dir.name
if destination.exists():
    raise FileExistsError(f'目的資料夾已存在，不覆蓋：{destination}')
shutil.copytree(output_dir, destination)
print('Drive folder ID:', DRIVE_FOLDER_ID)
print('Archived to:', destination)

In [ ]:
required = {
    'phase25_summary.md', 'phase25_run_metadata.json', 'phase25_candidate_signals.csv',
    'phase25_forward_horizon_results.csv', 'phase25_horizon_profile.csv',
    'phase25_prior5_controlled_regressions.csv', 'phase25_prior20_controlled_regressions.csv',
    'phase25_prior5_interactions.csv', 'phase25_prior20_interactions.csv',
    'phase25_effect_attenuation.csv', 'phase25_prior_return_descriptive.csv',
    'phase25_temporal_robustness.csv',
    'phase25_significant_results.csv', 'run_info.txt'
}
missing = required - {p.name for p in destination.iterdir()}
if missing:
    raise RuntimeError(f'Drive archive missing files: {sorted(missing)}')
print('Phase 2.5 archive verified:', destination)